# Task 03 – Human & Car Detection with Counting
## Inference Pipeline with Bounding Box Visualization

This notebook demonstrates:
1. Running detection on single images
2. Bounding box drawing (green=person, orange=car)
3. Human count overlay
4. Batch processing results
5. Count statistics across the test set

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

from src.detect import run_inference, count_objects, draw_detections, process_folder

WEIGHTS  = '../runs/detect/visdrone_yolov8n/weights/best.pt'
VAL_IMGS = '../data/VisDrone2019-DET-val/images'
OUT_DIR  = '../outputs/images'

model = YOLO(WEIGHTS)
print('Model loaded.')

In [ ]:
# ─── Single image inference demo ───
test_img = sorted(Path(VAL_IMGS).glob('*.jpg'))[0]
frame = cv2.imread(str(test_img))

detections = run_inference(model, frame)
counts     = count_objects(detections)
annotated  = draw_detections(frame, detections, counts)

print(f'Image: {test_img.name}')
print(f'Detections: {len(detections)}')
print(f'Persons: {counts["person"]}  |  Cars: {counts["car"]}')

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'Detection Output — 👤 {counts["person"]} persons  |  🚗 {counts["car"]} cars', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_single_detection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Batch processing on validation set ───
results = process_folder(model, VAL_IMGS, OUT_DIR)

# Summary stats
import numpy as np
persons = [r['person_count'] for r in results]
cars    = [r['car_count']    for r in results]

print(f'\n📊 Batch Results Summary')
print(f'  Images processed    : {len(results)}')
print(f'  Total persons found : {sum(persons)}')
print(f'  Total cars found    : {sum(cars)}')
print(f'  Avg persons/image   : {np.mean(persons):.1f}')
print(f'  Avg cars/image      : {np.mean(cars):.1f}')
print(f'  Max persons in img  : {max(persons)}')
print(f'  Avg inference time  : {np.mean([r["inference_ms"] for r in results]):.1f}ms')

In [ ]:
# ─── Visual grid of detections ───
val_imgs = sorted(Path(VAL_IMGS).glob('*.jpg'))[:9]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Detection Results – Validation Set Sample', fontsize=14, fontweight='bold')

CLASS_COLORS = {0: (0, 255, 80), 1: (0, 160, 255)}

for idx, img_path in enumerate(val_imgs):
    frame  = cv2.imread(str(img_path))
    dets   = run_inference(model, frame)
    counts = count_objects(dets)
    ann    = draw_detections(frame, dets, counts)

    ax = axes[idx // 3][idx % 3]
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'👤 {counts["person"]}  🚗 {counts["car"]}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_detection_grid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Count distribution plots ───
from src.visualize import plot_count_histogram
plot_count_histogram(results, save_path=f'{OUT_DIR}/03_count_histogram.png')

In [ ]:
# ─── Counting confidence: confidence score distribution ───
all_confs_person = []
all_confs_car    = []

for img_path in sorted(Path(VAL_IMGS).glob('*.jpg'))[:100]:
    frame = cv2.imread(str(img_path))
    dets  = run_inference(model, frame)
    for d in dets:
        if d['cls_id'] == 0: all_confs_person.append(d['conf'])
        else: all_confs_car.append(d['conf'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Detection Confidence Score Distribution', fontsize=12, fontweight='bold')

for ax, confs, label, color in [
    (axes[0], all_confs_person, 'Person', '#00C853'),
    (axes[1], all_confs_car,    'Car',    '#FF6D00'),
]:
    ax.hist(confs, bins=30, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(f'{label} (n={len(confs):,})')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.axvline(0.35, color='red', linestyle='--', label='threshold=0.35')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Counting Logic

The counting logic is intentionally simple and transparent:

```python
# Frame-level counting
def count_objects(detections):
    return {
        'person': sum(1 for d in detections if d['cls_id'] == 0),
        'car':    sum(1 for d in detections if d['cls_id'] == 1),
    }
```

**For video:** counting is per-frame (instantaneous). 
**With tracking (Task 04):** we count unique track IDs → avoids double-counting the same person across frames.

### ✅ Task 03 Complete